# Stable Diffusion 画像生成 — Google Colab 版

GPU が無い人でも、無料の Google Colab (T4 GPU) でこの画像生成アプリを使えます。

## 使い方（上から順にセルを実行）
1. **GPU を有効化**: メニュー →『ランタイム』→『ランタイムのタイプを変更』→ ハードウェアアクセラレータ = **T4 GPU** → 保存
2. 各セルを上から順に ▶ で実行
3. 最後のセルで表示される `https://xxxxx.gradio.live` を開く

## 重要な注意
- このノートは **画像生成のみ**（txt2img / img2img / ControlNet / IP-Adapter / Upscale / Inpaint / バッチ生成 など）の構成です。
- **モデル（.safetensors）は各自で用意**します（セル 5 でダウンロード or Drive からコピー）。
- 無料 Colab は数時間でセッションが切れます。生成物を残したい場合はセル 6（Drive 保存）を実行してください。

## 1. GPU 確認
`Tesla T4` などが表示されれば OK。`command not found` の場合はランタイムのタイプが GPU になっていません。

In [2]:
!nvidia-smi

Sun Jun 14 20:52:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive をマウント（任意）
Drive 上の自分のモデルを使う場合、または生成物を Drive に保存する場合に実行します。
URL からモデルをダウンロードするだけなら、このセルは省略しても構いません。

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. コードを取得（GitHub から clone）
既定では `MetAIra/Local_StableDiffusion` を clone します。
**リポジトリが public でないと clone は失敗します**（後述の「配布する人向け」を参照）。

In [5]:
# 公開リポジトリ URL（既定: MetAIra/Local_StableDiffusion）
REPO_URL = 'https://github.com/MetAIra/Local_StableDiffusion.git'
# ※ リポジトリが public でないと clone は失敗します
import os
APP_DIR = '/content/sd_app'
if not os.path.exists(os.path.join(APP_DIR, 'app.py')):
    !git clone $REPO_URL $APP_DIR
%cd $APP_DIR
print('Code dir:', os.getcwd())
assert os.path.exists('app.py'), 'app.py が見つかりません。REPO_URL とリポジトリの公開設定を確認してください。'

Cloning into '/content/sd_app'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/sd_app'
/content
Code dir: /content


AssertionError: app.py が見つかりません。REPO_URL とリポジトリの公開設定を確認してください。

## 4. 依存関係をインストール（数分）
画像生成に必要なライブラリだけを入れます（torch/numpy 等は Colab のものを使います）。

> pip の依存警告（dependency conflict）が出ても、たいてい無害です。
> もし後で `ImportError` が出たら『ランタイム → セッションを再起動』してセル 5 以降を再実行してください。

In [ ]:
!pip install -q -r colab/requirements-colab.txt
print('インストール完了')

## 5. モデルの置き場所を設定
モデルは Colab のローカル領域 `/content/models` に置きます（高速・Drive の mmap エラー回避のため）。
このセルを実行するとフォルダ構造が作られます:
```
/content/models/
  ├── model/      ← ここに .safetensors を入れる（必須）
  ├── VAE/        ← 任意
  ├── LoRA/       ← 任意
  └── Negative/   ← 任意
```

In [ ]:
import os

# 環境変数（config.py / app.py がこれを読む）。app を import する前に設定すること。
os.environ['BASE_MODEL_DIR'] = '/content/models'   # モデルのベースフォルダ（ローカル）
os.environ['HF_HOME'] = '/content/hf_cache'         # ControlNet 等の HF DL 先（セッション内）
os.environ['SD_FEATURE_SET'] = 'image'              # 画像生成のみ表示

for sub in ['model', 'VAE', 'LoRA', 'Negative']:
    os.makedirs(os.path.join('/content/models', sub), exist_ok=True)

print('BASE_MODEL_DIR =', os.environ['BASE_MODEL_DIR'])
print('model/ の中身:', os.listdir('/content/models/model'))

### 5-A. モデルを URL からダウンロード
Hugging Face や Civitai の `.safetensors` 直リンクを指定します。
（Civitai はアカウントの API トークンが必要な場合があります。その場合は URL の末尾に `?token=あなたのトークン` を付けます。）

In [ ]:
# === モデルを URL からダウンロード ===
MODEL_URL = ''                       # 例: 'https://huggingface.co/.../model.safetensors'
FILE_NAME = 'my_model.safetensors'   # 保存名（.safetensors）

if MODEL_URL:
    dest = f'/content/models/model/{FILE_NAME}'
    !wget --content-disposition -O "$dest" "$MODEL_URL"
    import os
    print('保存: {} ({:.1f} MB)'.format(dest, os.path.getsize(dest) / 1e6))
else:
    print('MODEL_URL が空です。URL を入れて実行してください。（または 5-B で Drive からコピー）')

### 5-B.（代替）自分の Google Drive にあるモデルをコピー
先に「2. Google Drive をマウント」を実行しておいてください。
`DRIVE_MODEL_DIR` を自分の `.safetensors` が入っているフォルダに変更します。

In [ ]:
import os, glob, shutil

DRIVE_MODEL_DIR = '/content/drive/MyDrive/SD_models/model'   # ← 自分のフォルダに変更

if os.path.isdir(DRIVE_MODEL_DIR):
    for f in glob.glob(os.path.join(DRIVE_MODEL_DIR, '*.safetensors')):
        dst = os.path.join('/content/models/model', os.path.basename(f))
        if not os.path.exists(dst):
            print('copy:', os.path.basename(f))
            shutil.copy(f, dst)
    print('完了。model/ の中身:', os.listdir('/content/models/model'))
else:
    print('フォルダが見つかりません:', DRIVE_MODEL_DIR)

## 6.（任意）生成物を Google Drive に保存
セッションが切れても画像を残すため、`output/` を Drive のフォルダに繋ぎます。
先に「2. Google Drive をマウント」を実行しておいてください。

In [ ]:
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/SD_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
if not os.path.exists('output'):
    os.symlink(DRIVE_OUTPUT, 'output')
    print('output ->', DRIVE_OUTPUT)
else:
    print('output は既に存在します:', os.path.realpath('output'))

## 7. アプリを起動
実行後に表示される **`Running on public URL: https://xxxxx.gradio.live`** を開いてください。
（モデルを追加した場合は、このセルを再実行する前に『ランタイム → セッションを再起動』が必要です。）

In [ ]:
from app import create_app

demo = create_app()
demo.launch(share=True)